In [1]:
#PVT chkPN
import pandas as pd
import numpy as np

In [2]:
#SARING SEMUA PN DI PVT CHKPN
# ================================
# Load main data
# ================================
dataPN_df = pd.read_excel("PTC Normal Draft.xlsx")

df = (
    dataPN_df[['PN AL78', 'PN used']]
    .drop_duplicates()
    .assign(**{
        "PN AL78": dataPN_df["PN AL78"].astype(str).str.strip()
    })
    .sort_values(
        by=["PN AL78", "PN used"],
        ascending=[True, True],
        na_position="last"
    )
    .reset_index(drop=True)
)


# Skip 1 column
df.insert(2, '', '')

# Ensure string
df['PN AL78'] = df['PN AL78'].astype(str)

# PN no suffix
df['PN no suffx'] = df['PN AL78'].apply(
    lambda x: x[:-2] if x.endswith('X') else x
)

In [3]:
# ================================
# Load PMF replacement file
# ================================
pmf_df = pd.read_excel("PMF19,20,41 17Jan2026.xlsx")

# Drop rows where PART NO. or CHG TO is missing
pmf_df = pmf_df.dropna(subset=['PART NO.', 'CHG TO'])

# Convert AFTER dropna
pmf_df['PART NO.'] = pmf_df['PART NO.'].astype(str).str.strip()
pmf_df['CHG TO'] = pmf_df['CHG TO'].astype(str).str.strip()

# Ensure unique lookup (Excel VLOOKUP = first match)
pmf_map = (
    pmf_df
    .drop_duplicates(subset='PART NO.', keep='first')
    .set_index('PART NO.')['CHG TO']
)


In [4]:
import numpy as np

def excel_vlookup(key, lookup_map):
    if pd.isna(key):
        return np.nan

    key = str(key).strip()

    # protect against "nan" string
    if key.lower() == "nan" or key == "":
        return np.nan

    val = lookup_map.get(key, np.nan)

    if pd.isna(val) or val == "0":
        return np.nan

    return val


In [5]:
df['Rplcmt'] = df['PN no suffx'].apply(lambda x: excel_vlookup(x, pmf_map))
df['NPN1']   = df['Rplcmt']
df['NPN2']   = df['NPN1'].apply(lambda x: excel_vlookup(x, pmf_map))
df['NPN3']   = df['NPN2'].apply(lambda x: excel_vlookup(x, pmf_map))
df['NPN4']   = df['PN AL78'].apply(
    lambda x: excel_vlookup(x, pmf_map) if str(x).endswith("X") else np.nan
)
df['NPN5']   = df['NPN4'].apply(lambda x: excel_vlookup(x, pmf_map))
df['PN Final'] = (
    df['NPN5']
    .combine_first(df['NPN4'])
    .combine_first(df['NPN3'])
    .combine_first(df['NPN2'])
    .combine_first(df['NPN1'])
    .combine_first(df['Rplcmt'])
    .combine_first(df['PN no suffx'])
)


In [6]:
print(df)

          PN AL78      PN used      PN no suffx Rplcmt NPN1  NPN2  NPN3  NPN4  \
0          107460    010746000           107460    NaN  NaN   NaN   NaN   NaN   
1          107695    010769500           107695    NaN  NaN   NaN   NaN   NaN   
2          108722    010872200           108722    NaN  NaN   NaN   NaN   NaN   
3          109319    010931900           109319    NaN  NaN   NaN   NaN   NaN   
4          109660    010966000           109660    NaN  NaN   NaN   NaN   NaN   
..            ...          ... ..           ...    ...  ...   ...   ...   ...   
981       S   658    S00065800          S   658    NaN  NaN   NaN   NaN   NaN   
982       S   962    S00096200          S   962    NaN  NaN   NaN   NaN   NaN   
983  S   962    E  S00096200 E     S   962    E    NaN  NaN   NaN   NaN   NaN   
984       S  1031    S00103100          S  1031    NaN  NaN   NaN   NaN   NaN   
985       S  2268    S00226800          S  2268    NaN  NaN   NaN   NaN   NaN   

     NPN5      PN Final  
0

In [6]:
df.to_excel("Pvt chkPN draft.xlsx", index=False)